# Precompute Random-Model Trajectories

The random-models widget shows ~100 networks per cell (init kind × width × depth). Running the gradient flow live in the browser is too slow, so we precompute the trajectories here and ship them as a compact data file the widget just loads and draws.

This is a faithful port of the dynamics in `dynamics.js`. For each cell we run `N_TRAJ` random networks from the standard width-normalized initialization, evolve each toward a fixed norm-1 whitened target, and record:

- the trajectory in function space, **projected into that model's own $p_0$–$p^\star$ plane** (the 2D slice the dynamics live in),
- `init` — did it **start** inside the sharpening ball, $p_0^\top(p_0-p^\star) < 0$,
- `ever` — did its sharpness **ever** rise above its starting value.

## Compact encoding

Plain JSON of 3-decimal floats is wasteful (~2.4 MB). Instead we:
1. **trim** each path to `MAX_PTS` points (a smooth curve needs few),
2. **quantize** coordinates to integers (×`SCALE`, so 0.961 → 96),
3. **delta-encode** along each path (smooth ⇒ consecutive deltas are tiny),
4. **split** paths / init flags / ever flags into parallel arrays.

This brings the file to ~0.4 MB of mostly single-digit integers (and far less gzipped), while staying a readable text file. The widget reverses these four steps on load.

## Configuration

In [ ]:
WIDTHS = [2, 3, 4, 5, 6]
DEPTHS = [1, 2, 3, 4, 5]
KINDS  = ["gaussian", "uniform"]
N_TRAJ = 100              # trajectories per cell (all are displayed)
MAX_PTS = 16             # points kept per trajectory after downsampling
SCALE = 100              # coordinate quantization (0.961 -> 96)
TARGET_NORM = 1.0

ETA = 4e-3
MAX_STEPS = 2500
LOSS_TOL = 1e-3
RECORD_EVERY = 4
SEED = 12345

import json, math, random
import numpy as np

## Dynamics (port of `dynamics.js`)

In [ ]:
def random_model(rng, kind, d, n):
    if kind == "gaussian":
        s = math.sqrt(1.0 / d)
        return ([rng.gauss(0.0, s) for _ in range(d)],
                [rng.gauss(0.0, 1.0) for _ in range(n)])
    aw, ab = math.sqrt(3.0 / d), math.sqrt(3.0)
    return ([rng.uniform(-aw, aw) for _ in range(d)],
            [rng.uniform(-ab, ab) for _ in range(n)])

def whitened_data(d, pstar):
    X, sd = [], math.sqrt(d)
    for j in range(d):
        e = [0.0]*d; e[j] = sd; X.append(e)
        e2 = [0.0]*d; e2[j] = -sd; X.append(e2)
    y = [sum(pstar[i]*xk[i] for i in range(d)) for xk in X]
    return np.array(X, float), np.array(y, float)

def run_trajectory(w1, b, X, y):
    """Explicit-Euler gradient flow. Returns recorded p(t) points and sharpness series."""
    w1 = np.array(w1, float); b = np.array(b, float); N = X.shape[0]
    sharp = lambda w1, b: float((np.prod(b)**2)*(1.0 + np.sum(1.0/b**2)*np.sum(w1**2)))
    pts = [(np.prod(b)*w1).copy()]
    sharps = [sharp(w1, b)]
    for step in range(MAX_STEPS):
        B = np.prod(b)
        wx = X @ w1
        r = B*wx - y
        loss = 0.5*np.mean(r*r)
        gw1 = (B/N)*(X.T @ r)
        gb = np.array([(1.0/N)*np.sum(r*wx*(B/bi)) for bi in b])
        w1 -= ETA*gw1; b -= ETA*gb
        if (step+1) % RECORD_EVERY == 0:
            pts.append((np.prod(b)*w1).copy()); sharps.append(sharp(w1, b))
        if loss < LOSS_TOL:
            break
    pts.append((np.prod(b)*w1).copy()); sharps.append(sharp(w1, b))
    return pts, sharps

## Project into the $p_0$–$p^\star$ plane, downsample, encode

Each trajectory is projected onto its own orthonormal basis $\{e_1=p^\star/\lVert p^\star\rVert,\ e_2\perp e_1\text{ toward }p_0\}$, trimmed to `MAX_PTS`, quantized to integers, then delta-encoded.

In [ ]:
def project(pts, pstar, sign=1):
    # e1 = p* direction; e2 = component of p0 orthogonal to e1, normalized.
    # e2's SIGN is a free choice (the plane is identical either way), so we pass
    # a random +/-1 per trajectory to scatter paths above AND below the axis,
    # rather than folding everything into the upper half-plane. This changes only
    # the cosmetic up/down placement, never a path's true norm or angle-to-target.
    d = len(pstar)
    pn = math.sqrt(sum(v*v for v in pstar))
    e1 = np.array([v/pn for v in pstar])
    p0 = np.array(pts[0])
    a = float(p0 @ e1)
    e2 = p0 - a*e1
    e2n = float(np.linalg.norm(e2))
    e2 = sign * e2/e2n if e2n > 1e-9 else np.zeros(d)
    return [[float(np.array(p) @ e1), float(np.array(p) @ e2) if e2n > 1e-9 else 0.0] for p in pts]

def downsample(pts, maxpts):
    if len(pts) <= maxpts:
        return pts
    stride = len(pts)/maxpts
    idx = sorted(set([int(i*stride) for i in range(maxpts)] + [len(pts)-1]))
    return [pts[i] for i in idx]

def delta_encode(pts2d):
    """quantize to ints (xSCALE), then delta-encode into a flat array."""
    q = [[round(x*SCALE), round(y*SCALE)] for x, y in pts2d]
    flat = [q[0][0], q[0][1]]
    for i in range(1, len(q)):
        flat.append(q[i][0]-q[i-1][0]); flat.append(q[i][1]-q[i-1][1])
    return flat

def init_sharpens(p0, pstar):
    return sum(p0[k]*(p0[k]-pstar[k]) for k in range(len(pstar))) < 0

def build_cell(kind, d, n):
    rng = random.Random(SEED)
    pstar = [0.0]*d; pstar[0] = TARGET_NORM
    X, y = whitened_data(d, pstar)
    paths, inits, evers = [], [], []
    for _ in range(N_TRAJ):
        w1, b = random_model(rng, kind, d, n)
        pts, sharps = run_trajectory(w1, b, X, y)
        sign = 1 if rng.random() < 0.5 else -1   # random half-plane per trajectory
        proj = downsample(project(pts, pstar, sign), MAX_PTS)
        paths.append(delta_encode(proj))
        inits.append(1 if init_sharpens([float(v) for v in pts[0]], pstar) else 0)
        evers.append(1 if max(sharps) > sharps[0]*1.001 else 0)
    return {"p": paths, "i": inits, "e": evers}

## Run the grid

~5 minutes for the full 50 cells × 100 trajectories. Set `N_TRAJ = 25` in the config cell to test the pipeline quickly first.

In [ ]:
data = {}
for kind in KINDS:
    data[kind] = {}
    for d in WIDTHS:
        data[kind][str(d)] = {}
        for n in DEPTHS:
            cell = build_cell(kind, d, n)
            data[kind][str(d)][str(n)] = cell
            ninit, never = sum(cell["i"]), sum(cell["e"])
            print(f"{kind:8s} d={d} n={n}: init {ninit}/{N_TRAJ}  ever {never}/{N_TRAJ}")
print("\ndone")

## Quick sanity check

Decode one path back and confirm it starts near the origin and ends near $p^\star=(1,0)$.

In [ ]:
def decode(flat):
    pts = []
    x, y = flat[0], flat[1]
    pts.append([x/SCALE, y/SCALE])
    for i in range(2, len(flat), 2):
        x += flat[i]; y += flat[i+1]
        pts.append([x/SCALE, y/SCALE])
    return pts

sample = decode(data["gaussian"]["2"]["3"]["p"][0])
print("first point:", sample[0], " last point:", sample[-1], " (last should be near [1, 0])")
print("points per path:", len(sample))

## Write `trajectories.js`

In [ ]:
import os
payload = {
    "meta": {"n_traj": N_TRAJ, "max_pts": MAX_PTS, "scale": SCALE,
             "widths": WIDTHS, "depths": DEPTHS, "kinds": KINDS, "target_norm": TARGET_NORM},
    "data": data,
}
header = (
    "// ============================================================\n"
    "// trajectories.js - PRECOMPUTED widget trajectories (generated).\n"
    "// Regenerate: run gen_trajectories.ipynb\n"
    "// Compact encoding (decoded by the widget on load):\n"
    "//   TRAJ_DATA.data[kind][d][n] = { p:[flat..], i:[0/1..], e:[0/1..] }\n"
    "//   each p[k] is a delta-encoded, integer-quantized (xSCALE) path;\n"
    "//   decode: x0,y0 = p[0],p[1]; then cumulative-sum the (dx,dy) pairs,\n"
    "//   divide by SCALE. i[k]/e[k] are the init/ever booleans for path k.\n"
    "// ============================================================\n\n"
    "export const TRAJ_DATA = "
)
with open("trajectories.js", "w") as f:
    f.write(header + json.dumps(payload, separators=(",", ":")) + ";\n")
print(f"wrote trajectories.js ({os.path.getsize('trajectories.js')/1e6:.2f} MB)")